In [1]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi
import os
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as Data


from sklearn.model_selection import LeaveOneOut
import time
from tqdm import tqdm
import numpy as np
import gc
import sys
sys.setrecursionlimit(50000)
import pickle
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
# from tensorboardX import SummaryWriter
torch.nn.Module.dump_patches = True
import copy
import pandas as pd
#then import my own modules
from GCN import save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight
from rdkit import Chem
# from rdkit.Chem import AllChem
from rdkit.Chem import QED
from rdkit.Chem import rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
%matplotlib inline
from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.cm as cm
import matplotlib
import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display
from torch.utils.data import Dataset, TensorDataset, DataLoader


import itertools
from sklearn.metrics import r2_score
import scipy
from itertools import combinations
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import pickle
import os
import time
from rdkit import Chem
from sklearn.model_selection import LeaveOneOut
from collections import defaultdict


device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 1.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 33.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 28.6 MB/s eta 0:00:0000:0100:01


/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


cuda


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Fingerprint(nn.Module):
    def __init__(self, radius, T, input_feature_dim, input_bond_dim,
                 fingerprint_dim, output_units_num, p_dropout):
        super(Fingerprint, self).__init__()
        
        self.radius = radius
        self.T = T
        
        # Atom embedding
        self.atom_fc = nn.Linear(input_feature_dim, fingerprint_dim)
        self.neighbor_fc = nn.Linear(input_feature_dim + input_bond_dim, fingerprint_dim)
        self.gru_cells = nn.ModuleList([nn.GRUCell(fingerprint_dim, fingerprint_dim) for _ in range(radius)])
        self.align_layers = nn.ModuleList([nn.Linear(2 * fingerprint_dim, 1) for _ in range(radius)])
        self.attend_layers = nn.ModuleList([nn.Linear(fingerprint_dim, fingerprint_dim) for _ in range(radius)])
        
        # Molecule embedding
        self.mol_gru_cell = nn.GRUCell(fingerprint_dim, fingerprint_dim)
        self.mol_align = nn.Linear(2 * fingerprint_dim, 1)
        self.mol_attend = nn.Linear(fingerprint_dim, fingerprint_dim)
        
        self.dropout = nn.Dropout(p=p_dropout)
        self.output = nn.Linear(fingerprint_dim, output_units_num)
        self.pairwise_output = nn.Linear(fingerprint_dim * 2, output_units_num)
        
    def forward(self, atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1,
                atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2):
        # Process first molecule
        atom_feature1, mol_feature1 = self.process_single_molecule(
            atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1
        )

        # Process second molecule
        atom_feature2, mol_feature2 = self.process_single_molecule(
            atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2
        )

        # Concatenate molecular features
        combined_feature = torch.cat([mol_feature1, mol_feature2], dim=1)

        # Final pairwise prediction
        pairwise_prediction = self.pairwise_output(self.dropout(combined_feature))

        return atom_feature1, atom_feature2, pairwise_prediction

    def process_single_molecule(self, atom_list, bond_list, atom_degree_list, bond_degree_list, atom_mask):
        batch_size, mol_length, num_atom_feat = atom_list.size()
        atom_mask = atom_mask.unsqueeze(2)

        atom_feature = F.leaky_relu(self.atom_fc(atom_list))
        neighbor_feature = self.get_neighbor_feature(atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size)

        attend_mask, softmax_mask = self.create_masks(atom_degree_list, mol_length)

        atom_feature = self.apply_atom_attention(atom_feature, neighbor_feature, attend_mask, softmax_mask)
        mol_feature = torch.sum(F.relu(atom_feature) * atom_mask, dim=-2)

        mol_feature = self.apply_molecule_attention(atom_feature, mol_feature, atom_mask)

        return atom_feature, mol_feature


    def get_neighbor_feature(self, atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size):
        bond_neighbor = torch.stack([bond_list[i][bond_degree_list[i]] for i in range(batch_size)], dim=0)
        atom_neighbor = torch.stack([atom_list[i][atom_degree_list[i]] for i in range(batch_size)], dim=0)
        neighbor_feature = torch.cat([atom_neighbor, bond_neighbor], dim=-1)
        return F.leaky_relu(self.neighbor_fc(neighbor_feature))

    def create_masks(self, atom_degree_list, mol_length):
        attend_mask = (atom_degree_list != mol_length - 1).float().unsqueeze(-1)
        softmax_mask = torch.where(atom_degree_list == mol_length - 1, torch.tensor(-9e8).cuda(), torch.tensor(0.).cuda()).unsqueeze(-1)
        return attend_mask, softmax_mask

    def apply_atom_attention(self, atom_feature, neighbor_feature, attend_mask, softmax_mask):
        #print("Applying atom-level attention")
        batch_size, mol_length = atom_feature.shape[:2]
        
        for d in range(self.radius):
            atom_feature_expand = atom_feature.unsqueeze(-2).expand(-1, -1, neighbor_feature.size(2), -1)
            feature_align = torch.cat([atom_feature_expand, neighbor_feature], dim=-1)
            
            align_score = F.leaky_relu(self.align_layers[d](self.dropout(feature_align))) + softmax_mask
            attention_weight = F.softmax(align_score, -2) * attend_mask
            
            context = torch.sum(attention_weight * self.attend_layers[d](self.dropout(neighbor_feature)), -2)
            context = F.elu(context)
            
            atom_feature = self.gru_cells[d](
                context.view(batch_size * mol_length, -1),
                atom_feature.view(batch_size * mol_length, -1)
            ).view(batch_size, mol_length, -1)
            
            neighbor_feature = F.relu(atom_feature).unsqueeze(-2).expand(-1, -1, neighbor_feature.size(2), -1)
        
        return atom_feature

    def apply_molecule_attention(self, atom_feature, mol_feature, atom_mask):
        #print("Applying mol-level attention")
        batch_size, mol_length = atom_feature.shape[:2]
        mol_softmax_mask = torch.where(atom_mask.squeeze() == 0, torch.tensor(-9e8).cuda(), torch.tensor(0.).cuda())
        
        for _ in range(self.T):
            mol_prediction_expand = mol_feature.unsqueeze(-2).expand(-1, mol_length, -1)
            mol_align = torch.cat([mol_prediction_expand, atom_feature], dim=-1)
            mol_align_score = F.leaky_relu(self.mol_align(mol_align)) + mol_softmax_mask.unsqueeze(-1)
            mol_attention_weight = F.softmax(mol_align_score, -2) * atom_mask
            
            mol_context = torch.sum(mol_attention_weight * self.mol_attend(self.dropout(atom_feature)), -2)
            mol_context = F.elu(mol_context)
            mol_feature = self.mol_gru_cell(mol_context, mol_feature)
        
        return mol_feature



class RelativeGCNModel(nn.Module):
    def __init__(self, radius, T, input_feature_dim, input_bond_dim,
                 fingerprint_dim, output_units_num, p_dropout):
        super(RelativeGCNModel, self).__init__()
        
        self.radius = radius
        self.T = T
        

        hidden_dim = fingerprint_dim * 4  
        self.atom_fc = nn.Sequential(
            nn.Linear(input_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        
        self.neighbor_fc = nn.Sequential(
            nn.Linear(input_feature_dim + input_bond_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        

        self.gcn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fingerprint_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, fingerprint_dim)
            ) for _ in range(radius * 2)  
        ])
        
        self.dropout = nn.Dropout(p=p_dropout)
        
        # More complex output layer for pairwise prediction
        self.output = nn.Sequential(
            nn.Linear(fingerprint_dim * 2, hidden_dim),  # Input is concatenated features of two molecules
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_units_num)
        )

        
    def forward(self, atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1,
                atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2):
        
        batch_size, mol_length, num_atom_feat = atom_list1.size()

        # Process first molecule
        atom_feature1, mol_feature1 = self.process_molecule(atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1)
        
        # Process second molecule
        atom_feature2, mol_feature2 = self.process_molecule(atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2)
        
        # Concatenate features of both molecules and predict pairwise difference
        pairwise_feature = torch.cat([mol_feature1, mol_feature2], dim=1)
        pairwise_prediction = self.output(self.dropout(pairwise_feature))
        
        return atom_feature1, atom_feature2, pairwise_prediction

    
    
    def process_molecule(self, atom_list, bond_list, atom_degree_list, bond_degree_list, atom_mask):
        batch_size, mol_length, num_atom_feat = atom_list.size()
        atom_mask = atom_mask.unsqueeze(2)
        

        atom_feature = self.atom_fc(atom_list)
        
        neighbor_feature = self.get_neighbor_feature(atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length)
        
        for i, layer in enumerate(self.gcn_layers):
            neighbor_feature = layer(neighbor_feature)
            atom_feature = atom_feature + neighbor_feature
            atom_feature = self.dropout(atom_feature)
        
        mol_feature = torch.sum(F.relu(atom_feature) * atom_mask, dim=1)
        return atom_feature, mol_feature

    

    def get_neighbor_feature(self, atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length):
        neighbor_feature = []
        for i in range(batch_size):
            atom_degrees = atom_degree_list[i]
            bond_degrees = bond_degree_list[i]
        
            # Ensure indices are within bounds
            atom_degrees = torch.clamp(atom_degrees, 0, mol_length - 1)
            bond_degrees = torch.clamp(bond_degrees, 0, bond_list.size(1) - 1)
        
        # No need to pad, use the original tensors
            atom_features = atom_list[i]
            bond_features = bond_list[i]
        
        # Gather neighbor features
            neighbor_atoms = atom_features[atom_degrees]
            neighbor_bonds = bond_features[bond_degrees]
            
        # Combine atom and bond features
            mol_neighbor_feature = torch.cat([neighbor_atoms, neighbor_bonds], dim=-1)
        
        # Apply neighbor_fc to each atom's neighborhood and sum
            mol_neighbor_feature = self.neighbor_fc(mol_neighbor_feature)
            mol_neighbor_feature = mol_neighbor_feature.sum(dim=1)
        
            neighbor_feature.append(mol_neighbor_feature)
    
        neighbor_feature = torch.stack(neighbor_feature, dim=0)
        return F.relu(neighbor_feature)
    


In [3]:
def custom_cv_split(df):
    """Generate train/test splits based on unique SMILES compounds."""
    # Extract all unique SMILES from the dataset
    all_smiles = set(smiles for pair in df['SMILES_pair'] for smiles in pair.split(','))
    
    for test_smile in all_smiles:
        # Create indices for proper train/test split
        test_indices = []
        train_indices = []
        
        for idx, pair in enumerate(df['SMILES_pair']):
            smiles1, smiles2 = pair.split(',')
            # Only put pairs in test set where test_smile appears
            if test_smile in (smiles1, smiles2):
                test_indices.append(idx)
            else:
                train_indices.append(idx)
        
        yield train_indices, test_indices, test_smile


def train_and_evaluate(model, df, feature_dicts, optimizer, loss_function, num_epochs=800, batch_size=38, patience=30):
    """Train model using leave-one-compound-out cross-validation with proper data isolation."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    fold_results = []
    all_true_values = []
    all_predictions = []
    final_dic = {}
    
    # Save initial model state for resetting between folds
    initial_state = {k: v.clone() for k, v in model.state_dict().items()}
    n_splits = len(set(smiles for pair in df['SMILES_pair'] for smiles in pair.split(',')))
    
    for fold, (train_val_index, test_index, test_smile) in enumerate(custom_cv_split(df), 1):
        print(f"\nFold {fold}/{n_splits} - Testing SMILES: {test_smile}")
        
        # Reset model and optimizer for each fold
        model.load_state_dict(initial_state) 
        optimizer = type(optimizer)(model.parameters(), **optimizer.defaults)
        
        # Split data - ensure test compound is completely isolated
        train_val_df = df.iloc[train_val_index].copy()
        test_df = df.iloc[test_index].copy()
        
        # Further split training data into train and validation
        # Ensure validation set doesn't contain the test compound
        train_smiles = set(smiles for pair in train_val_df['SMILES_pair'] for smiles in pair.split(','))
        if test_smile in train_smiles:
            print(f"ERROR: Test SMILES found in training set! Skipping fold.")
            continue
            
        train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)
        
        # Verify validation set doesn't contain test compound
        val_smiles = set(smiles for pair in val_df['SMILES_pair'] for smiles in pair.split(','))
        if test_smile in val_smiles:
            print(f"ERROR: Test SMILES found in validation set! Skipping fold.")
            continue
        
        # Training loop
        best_val_loss = float('inf')
        epochs_no_improve = 0
        best_model_state = None
        
        for epoch in range(num_epochs):
            # Train
            model.train()
            total_loss = 0
            train_df = train_df.sample(frac=1).reset_index(drop=True)  # Shuffle
            
            # Process in batches to manage memory
            for i in range(0, len(train_df), batch_size):
                batch_df = train_df.iloc[i:i+batch_size]
                smile_pairs = batch_df.SMILES_pair.values
                y_val = batch_df[targets].values
                
                # Process SMILES pairs
                smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
                
                # Get molecular features
                x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
                x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
                
                # Convert to tensors
                x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
                x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
                x_mask1 = torch.Tensor(x_mask1).to(device)
                
                x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
                x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
                x_mask2 = torch.Tensor(x_mask2).to(device)
                
                # Forward and backward pass
                optimizer.zero_grad()
                output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                                   x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
                predictions = output_tuple[2]
                
                loss = loss_function(predictions, torch.Tensor(y_val).to(device))
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / (len(train_df) // batch_size + 1)
            
            # Validation
            val_loss, val_r2 = validate(model, val_df, feature_dicts, loss_function, device, batch_size)
            print(f'Epoch {epoch+1}/{num_epochs}: Train Loss={avg_loss:.4f}, Val Loss={val_loss:.4f}, Val R2={val_r2:.4f}')
            
            # Early stopping check
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve == patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break
        
        # Load best model for evaluation
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        
        # Evaluate on test set
        test_loss, test_r2, true_values, predictions, result_dict = validate(
            model, test_df, feature_dicts, loss_function, device, batch_size, 
            return_predictions=True, test_smile=test_smile
        )
        
        print(f'Fold {fold} results: Val Loss={best_val_loss:.4f}, Test Loss={test_loss:.4f}, Test R2={test_r2:.4f}')
        
        final_dic.update(result_dict)
        all_true_values.extend(true_values)
        all_predictions.extend(predictions)
        fold_results.append((best_val_loss, test_loss, test_r2))
    
    # Summarize results
    if fold_results:
        val_losses, test_losses, test_r2s = zip(*fold_results)
        # print('\nCross-validation summary:')
        # print(f'Average validation loss: {np.mean(val_losses):.4f}')
        # print(f'Average test loss: {np.mean(test_losses):.4f}')
        # print(f'Average test R2: {np.mean(test_r2s):.4f}')

        overall_r2 = r2_score(all_true_values, all_predictions)
        # print(f'Overall R2: {overall_r2:.4f}')
        
        return fold_results, overall_r2, final_dic
    else:
        # print("No valid folds completed.")
        return [], 0.0, {}


    
    
def prepare_model_and_data_for_relative_learning(raw_filename, targets=None, p_dropout=0.1, fingerprint_dim=150, 
                                              weight_decay=2, learning_rate=3, radius=3, T=2):
    """Prepare data and model for relative learning approach with memory-efficient implementation."""
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    # Load and preprocess data
    feature_filename = raw_filename.replace('.csv', '.pickle')
    smiles_targets_df = pd.read_csv(raw_filename)
    smilesList = smiles_targets_df.SMILES.values
    
    # Process SMILES strings
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            pass
    
    # Filter dataframe and add canonical SMILES
    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    # Load or generate feature dictionaries
    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = get_smiles_dicts(smilesList)

    # Filter by available features
    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    # Create pairs and calculate relative targets more efficiently
    # Use batch processing to avoid memory issues
    batch_size = 1000  # Adjust based on available memory
    n_samples = len(remained_df)
    smile_pairs = []
    relative_targets = []
    
    # Process in batches to manage memory
    for i in range(0, n_samples, batch_size):
        batch_end = min(i + batch_size, n_samples)
        batch_df = remained_df.iloc[i:batch_end]
        
        for idx1, row1 in batch_df.iterrows():
            # For each compound in the batch, compare with all other compounds
            # This avoids the n² complexity by processing in smaller chunks
            for idx2, row2 in remained_df.iterrows():
                if idx1 != idx2:  # Don't compare with self
                    smile1 = row1['cano_smiles']
                    smile2 = row2['cano_smiles']
                    target1 = row1[targets].values
                    target2 = row2[targets].values
                    
                    rel_target = target1 - target2
                    smile_pairs.append(f"{smile1},{smile2}")
                    relative_targets.append(rel_target)
                    
                    # Optional: limit the number of pairs per compound to reduce dataset size
                    # if len(smile_pairs) % 1000 == 0:
                    #     print(f"Generated {len(smile_pairs)} pairs...")

    # Create dataframe with pairs and targets
    df = pd.DataFrame(relative_targets, columns=targets)
    df['SMILES_pair'] = smile_pairs
    df = df[['SMILES_pair'] + targets]

    # Get feature dimensions from first pair
    smile1, smile2 = smile_pairs[0].split(',')
    x_atom1, x_bonds1, _, _, _, _ = get_smiles_array([smile1], feature_dicts)
    num_atom_features = x_atom1.shape[-1]
    num_bond_features = x_bonds1.shape[-1]

    # Initialize model and optimizer
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loss_function = nn.MSELoss()
    model = RelativeGCNModel(radius, T, num_atom_features, num_bond_features, fingerprint_dim, len(targets), p_dropout)
    model.to(device)
    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

    return model, optimizer, loss_function, df, feature_dicts, remained_df

def validate(model, val_df, feature_dicts, loss_function, device, batch_size, return_predictions=False, test_smile=None):
    """Validate model on validation or test data."""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_true_values = []
    result_dict = {}

    with torch.no_grad():
        for i in range(0, len(val_df), batch_size):
            batch_df = val_df.iloc[i:i+batch_size]
            smile_pairs = batch_df.SMILES_pair.values
            y_val = batch_df[targets].values
            
            # Process SMILES pairs
            smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
            
            # Get molecular features
            x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
            x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
            
            # Convert to tensors
            x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
            x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
            x_mask1 = torch.Tensor(x_mask1).to(device)
            
            x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
            x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
            x_mask2 = torch.Tensor(x_mask2).to(device)
            
            # Forward pass
            output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                               x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
            predictions = output_tuple[2]
            
            # Calculate loss
            loss = loss_function(predictions, torch.Tensor(y_val).to(device))
            total_loss += loss.item()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_true_values.extend(y_val)

            # Store results for test SMILES
            if test_smile and return_predictions:
                for j, smile_pair in enumerate(smile_pairs):
                    smile1, smile2 = smile_pair.split(',')
                    if test_smile in (smile1, smile2):
                        if test_smile not in result_dict:
                            result_dict[test_smile] = {}
                        result_dict[test_smile][smile_pair] = {
                            target: {"actual": y_val[j][k], "predicted": predictions[j][k].item()}
                            for k, target in enumerate(targets)
                        }
    
    avg_loss = total_loss / (len(val_df) // batch_size + 1)
    r2 = r2_score(all_true_values, all_predictions)
    
    if return_predictions:
        return avg_loss, r2, all_true_values, all_predictions, result_dict
    return avg_loss, r2


def process_dictionary(df, unprocessed_dict, relative_is_counter_minus_host=True):
    """
    Process the unprocessed dictionary to calculate absolute predictions for each host.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing host information with columns 'Host', 'cano_smile', and target columns
    unprocessed_dict : dict
        Dictionary with hosts as main keys, smile pairs as subkeys, and target predictions
    relative_is_counter_minus_host : bool, default=True
        If True, relative prediction is (counter - host)
        If False, relative prediction is (host - counter)
    
    Returns:
    --------
    dict
        Processed dictionary with average absolute predictions and real values for each host and target
    """
    processed_dict = {}
    
    for host, smile_pairs in unprocessed_dict.items():
        host_row = df[df['Host'] == host].iloc[0]  # Get host's row once
        host_smile = host_row['cano_smiles']
        processed_dict[host] = {}
        
        for smile_pair, targets in smile_pairs.items():
            smile1, smile2 = map(str.strip, smile_pair.split(','))
            
            # Identify counter smile
            if smile1 == host_smile:
                counter_smile = smile2
            elif smile2 == host_smile:
                counter_smile = smile1
            else:
                continue  # Skip non-matching pairs
                
            counter_row = df[df['cano_smiles'] == counter_smile]
            if counter_row.empty:
                continue
                
            for target_name, values in targets.items():
                if target_name not in processed_dict[host]:
                    processed_dict[host][target_name] = {'predictions': []}
                
                counter_actual = counter_row[target_name].values[0]
                relative_pred = values['predicted']
                
                # Calculate absolute prediction
                host_abs_pred = (counter_actual - relative_pred) if relative_is_counter_minus_host else (relative_pred + counter_actual)
                processed_dict[host][target_name]['predictions'].append(host_abs_pred)
        
        # Add averages and real values
        for target_name in list(processed_dict[host].keys()):
            predictions = processed_dict[host][target_name]['predictions']
            if predictions:
                processed_dict[host][target_name] = {
                    'actual': host_row[target_name],
                    'predicted': sum(predictions) / len(predictions)

                }
            else:
                del processed_dict[host][target_name]
    
    return processed_dict


In [4]:
# Define targets
targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

# Prepare model and data
model, optimizer, loss_function, df, feature_dicts, remained_df = prepare_model_and_data_for_relative_learning(
    '/notebooks/Codebase/Database/calix smiles small set.csv'
)

# Train and evaluate model
fold_results, overall_r2, final_dic = train_and_evaluate(
    model, df, feature_dicts, optimizer, loss_function, 
    num_epochs=800, batch_size=32, patience=30
)



Fold 1/28 - Testing SMILES: O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)C2
Epoch 1/800: Train Loss=4.9917, Val Loss=4.5437, Val R2=0.0104
Epoch 2/800: Train Loss=4.4501, Val Loss=3.9499, Val R2=0.1542
Epoch 3/800: Train Loss=4.0440, Val Loss=3.1125, Val R2=0.3023
Epoch 4/800: Train Loss=3.3014, Val Loss=3.8167, Val R2=0.1671
Epoch 5/800: Train Loss=3.4465, Val Loss=2.4657, Val R2=0.4611
Epoch 6/800: Train Loss=2.5858, Val Loss=2.3773, Val R2=0.4883
Epoch 7/800: Train Loss=2.2302, Val Loss=2.5194, Val R2=0.4642
Epoch 8/800: Train Loss=2.0975, Val Loss=2.0440, Val R2=0.5547
Epoch 9/800: Train Loss=1.9012, Val Loss=2.1976, Val R2=0.5375
Epoch 10/800: Train Loss=1.8563, Val Loss=1.9633, Val R2=0.5862
Epoch 11/800: Train Loss=1.6280, Val Loss=1.9810, Val R2=0.5699
Epoch 12/800: Train Loss=1.4956, Val Loss=2.1229, Val R2=0.5416
Epoch 13/800: Train Loss=1.4810, Val Loss=1.7715, Val R2=0.6091
Epoch 14/800: Train Loss=1.5009, Val Loss=1.8579,

In [5]:
# Create a mapping dictionary from 'cano_smiles' to 'Host'
mapping = dict(zip(remained_df['cano_smiles'], remained_df['Host']))

# Create a new dictionary with updated keys
new_final_dic = {mapping.get(key, key): value for key, value in final_dic.items()}

# Replace the original dictionary with the new one
print(new_final_dic.keys())

abs_dict =process_dictionary(remained_df, new_final_dic, relative_is_counter_minus_host=True)

dict_keys(['PSC4', 'AH3', 'AP8', 'AP3', 'E3', 'AO1', 'AP1', 'AM1', 'AO3', 'AH4', 'P-NO2', 'E8', 'AP4', 'AP9', 'E6', 'AH5', 'AM2', 'AH7', 'E1', 'AP5', 'E7', 'AO2', 'AP7', 'AH2', 'AH6', 'AH1', 'AP6', 'E11'])


In [6]:
abs_dict

{'PSC4': {'H3K4': {'actual': 1.410986974, 'predicted': 1.2524323420101988},
  'H3K4ac': {'actual': 2.833213344, 'predicted': 2.143238232696538},
  'H3K4me1': {'actual': 0.78845736, 'predicted': 0.03630598968914184},
  'H3K4me2': {'actual': -0.186329578, 'predicted': -1.6254106211326418},
  'H3K4me3': {'actual': -0.579818495, 'predicted': -1.7953498241493004},
  'H3K9me3': {'actual': 0.336472237, 'predicted': -0.7920366991485935},
  'H3R2me2a': {'actual': 0.832909123, 'predicted': -0.41844788815991774},
  'H3R2me2s': {'actual': 1.740466175, 'predicted': 0.5037247805184448}},
 'AH3': {'H3K4': {'actual': 0.262364264, 'predicted': 1.1702853183600785},
  'H3K4ac': {'actual': 1.098612289, 'predicted': 2.0247206430432136},
  'H3K4me1': {'actual': -1.560647748, 'predicted': 0.06865736827537626},
  'H3K4me2': {'actual': -3.9633163, 'predicted': -1.5547842140951849},
  'H3K4me3': {'actual': -2.900422094, 'predicted': -1.7908436344050973},
  'H3K9me3': {'actual': -1.660731207, 'predicted': -0.846

In [7]:
with open ('/notebooks/Codebase/Result_dict/LOO GCN relative Small Dataset.pkl','wb') as f:
    pickle.dump(abs_dict, f)